## 本部分内容为grid2op环境的构建与注册，如无特殊需要请勿更改

测试使用l2rpn_wcci_2022环境，该环境中包含所有grid2op当前版本所支持的组件类型

In [1]:
import sys

# 导入grid2op必要组件
import grid2op
from grid2op.PlotGrid import PlotMatplot
from grid2op.Action import PlayableAction
from grid2op.Rules import AlwaysLegal
# from grid2op.Parameters import Parameters

# 导入自定义组件，包含日志以及注册环境所需全局变量
from registry_object import EnvironmentManager
from src.utils import get_logger

# 以下为在notebook中测试使用
import numpy as np
from typing import Dict, List, Any, Optional

[2025-03-04 16:04:53,153] [src.utils] [__init__.py(39)] [INFO] 这是带时间戳的日志文件测试信息。


In [2]:
# 设置环境名称与规则
# env_name = "l2rpn_case14_sandbox"
env_name = "l2rpn_wcci_2022" # 有存储单元

rules = AlwaysLegal()

# 初始化日志
logger = get_logger(__name__)

# 创建环境管理器实例用于注册并传递geid2op环境与观测实例
env_manager = EnvironmentManager()

try:
    from lightsim2grid import LightSimBackend
    bk_cls = LightSimBackend
    logger.info("成功导入 LightSimBackend 用于更快的后端计算。")
except ImportError as exc:
    logger.warning(f"导入 LightSimBackend 时发生错误：{exc}，将使用 PandaPowerBackend 作为后备方案。")
    from grid2op.Backend import PandaPowerBackend
    bk_cls = PandaPowerBackend

# 生成唯一环境标识符
env_id = "test1"

# 创建环境
env = grid2op.make(env_name,
                   action_class = PlayableAction,
                   gamerules_class = rules,
                   backend = bk_cls())
logger.info(f"环境 '{env_name}' 创建成功，使用后端: {bk_cls.__name__}。")

# 使用上下文管理器注册环境并处理相关操作
try:
    # 使用上下文管理器注册环境
    with env_manager.register_env(env_id, env):
        try:
            logger.info(f"环境 '{env_name}' 注册成功。")

            # 重置环境并获取初始观察值
            obs = env.reset()

            # 注册初始观察值
            env_manager.record_observation(env_id, obs)
            logger.info(f"环境 '{env_id}' 注册成功并获取初始观察值。")
        except Exception as e:
            logger.error(f"环境 '{env_name}' 注册失败，错误信息：{e}")
            sys.exit(1)

except Exception as e:
    logger.error(f"操作失败，错误信息：{e}")
    sys.exit(1)


[2025-03-04 16:04:53,171] [__main__] [2520142286.py(16)] [INFO] 成功导入 LightSimBackend 用于更快的后端计算。
[2025-03-04 16:04:53,418] [pandapower.convert_format] [convert_format.py(102)] [INFO] These dtypes could not be corrected: {'trafo': ['tap_min', 'tap_max']}
[2025-03-04 16:04:54,697] [__main__] [2520142286.py(30)] [INFO] 环境 'l2rpn_wcci_2022' 创建成功，使用后端: LightSimBackend。
[2025-03-04 16:04:54,699] [__main__] [2520142286.py(37)] [INFO] 环境 'l2rpn_wcci_2022' 注册成功。
[2025-03-04 16:04:55,140] [__main__] [2520142286.py(44)] [INFO] 环境 'test1' 注册成功并获取初始观察值。


## 在此行以后进行必要的功能测试

In [3]:
# from tool_impl_list import redispatch_impl, cancel_redispatch_impl

# gen_id_re = [13]
# amount_re = [3]


# redispatch_impl(env_manager, env_id, gen_id_re, amount_re)

In [5]:
'''
测试impl函数
'''

def storage_p_impl(env_manager, env_id: str, storage_id: List[int], amount: List[float]) -> Dict[str, Any]:
    """
    在grid2op环境中设置储能单元的充放电功率。

    参数:
    env_manager: 环境管理器实例，用于管理和注册环境。
    env_id (str): 环境实例的ID。
    storage_id (List[int]): 储能单元ID列表。
    amount (List[float]): 每个储能单元对应的充放电功率(MW)，正值表示充电(从电网吸收能量)，
                          负值表示放电(向电网释放能量)。

    返回:
    Dict[str, Any]: 包含状态和消息的字典。状态可以是 "success" 或 "failure"，消息描述了操作的结果。

    异常:
    ValueError: 如果环境ID无效或动作是模糊的。
    """
    try:
        logger.info(f"开始执行 storage_p 操作，env_id: {env_id}, storage_id: {storage_id}, amount: {amount}")
        
        # 获取环境实例
        env = env_manager._envs.get(env_id)
        if env is None:
            raise ValueError(f"无法找到 env_id 为 {env_id} 的环境实例。")
        
        # 创建动作
        act = env.action_space()
        act.storage_p = list(zip(storage_id, amount))
        print(act)  # 测试用，实装时删除
        
        # 检查动作是否模糊
        if act.is_ambiguous()[0]:
            ambiguity = act.is_ambiguous()[1]
            raise ValueError(f"动作是模糊的，无法执行。{ambiguity}")
        else:
            # 执行动作并获取新的观察
            obs, _, _, _ = env.step(act)
            
            # 使用 EnvironmentManager 注册新观察
            env_manager.record_observation(env_id, obs)
            
            logger.info(f"成功进行“储能单元充放电”操作。")
            return {
                "status": "success",
                "message": f"成功进行“储能单元充放电”操作。"
            }
    except Exception as e:
        logger.error(f"storage_p_impl 执行失败: {str(e)}")
        return {
            "status": "failure",
            "message": f"storage_p_impl 执行失败: {str(e)}"
        }

In [6]:
storage_p_impl(env_manager, env_id, [0, 1], [1, -1])

[2025-03-04 16:05:31,299] [__main__] [1184773345.py(23)] [INFO] 开始执行 storage_p 操作，env_id: test1, storage_id: [0, 1], amount: [1, -1]
[2025-03-04 16:05:31,310] [__main__] [1184773345.py(46)] [INFO] 成功进行“储能单元充放电”操作。


This action will:
	 - NOT change anything to the injections
	 - NOT perform any redispatching action
	 - Modify the storage units in the following way:
	 	 - Ask unit "storage_22_0" to absorb 1.00 MW (setpoint: 1.00 MW)
	 	 - Ask unit "storage_41_1" to produce 1.00 MW (setpoint: -1.00 MW)
	 - NOT perform any curtailment
	 - NOT force any line status
	 - NOT switch any line status
	 - NOT switch anything in the topology
	 - NOT force any particular bus configuration


{'status': 'success', 'message': '成功进行“储能单元充放电”操作。'}

In [ ]:
'''
在此block中进行初始观测
'''

# 以下根据需求修改
# obs_before = env_manager.get_latest_obs("test1")
# obs_before.line_status[:3]

'\n在此block中进行初始观测\n'

In [ ]:
'''
在此block中验证函数是否正确对组件进行更改
'''

# 以下根据需求修改
# obs_after = env_manager.get_latest_obs("test1")
# obs_after.line_status[:3]

'\n在此block中验证函数是否正确对组件进行更改\n'

In [ ]:
'''
绘制前后状态的观测图
'''

# plot_helper = PlotMatplot(env.observation_space)
# fig_before = plot_helper.plot_obs(obs_before)
# fig_after = plot_helper.plot_obs(obs_after)
# fig_before.show()
# fig_after.show()

'\n绘制前后状态的观测图\n'